## homework 4

In [12]:
import torch
import numpy as np
import pandas as pd
import yfinance

import pickle
import time

from torch.nn import Sequential
from torch.nn import GRU, LSTM
from torch.utils.data import DataLoader, TensorDataset
from torch import tensor

from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import shuffle

1. download and preprocess the data
- sliding window to collect sequences


TODO may need to go back and check for bad values that are not null. 

In [ ]:
# do this here if needed... download for 2021
# 80/20 split for data

M = 60
N = 1


def preprocess(data : pd.DataFrame, batch_size_ = 10):
    data_close = data["Close"].to_numpy()
    print("data_close size: ", np.shape(data_close))
    X_seq = []
    y_seq = []
    seqs = []

    # use sliding window to create sequences
    for i in range(len(data_close) - M - N):
        X_seq.append(data_close[i : i + M])
        y_seq.append(data_close[i + M : i + M + N])

    # remove extra dimension placed in. 
    X_seq_np = np.array(X_seq).squeeze()
    y_seq_np = np.array(y_seq).squeeze()

    # print(np.shape(X_seq_np))
    # print(np.shape(y_seq_np))

    # use sklearn to shuffle both and keep indices the same
    X_shuffle, y_shuffle = shuffle(X_seq_np, y_seq_np)

    # split 80/20
    split_index = int(np.size(X_shuffle, axis = 0) * 0.8)
    X_train = X_shuffle[:split_index - 1, :]
    X_test = X_shuffle[split_index:]
    y_train = y_shuffle[:split_index - 1]
    y_test = y_shuffle[split_index:]

    # Scale the data
    mms = MinMaxScaler()
    mms.fit_transform(X_train)
    mms.transform(X_test)

    # convert to tensors 
    X_train_tensor = torch.tensor(X_train, dtype = torch.float64)
    y_train_tensor = torch.tensor(y_train, dtype = torch.float64)
    X_test_tensor = torch.tensor(X_test, dtype = torch.float64)
    y_test_tensor = torch.tensor(y_test, dtype = torch.float64)

    # put inside tensor datasets
    train_set = TensorDataset(X_train_tensor, y_train_tensor)
    test_set = TensorDataset(X_test_tensor, y_test_tensor)

    # use data loaders to batch data. 
    load_train_set = DataLoader(train_set, batch_size= batch_size_)
    load_test_set = DataLoader(test_set, batch_size= batch_size_)

    return [load_train_set, load_test_set]


data_nvda = yfinance.download('NVDA', start = '2021-01-01', end = '2021-12-31')
data_gme = yfinance.download('GME', start = '2021-01-01', end = '2021-12-31')
data_dji = yfinance.download('DJI', start = '2021-01-01', end = '2021-12-31')
data_ma = yfinance.download('MA', start = '2021-01-01', end = '2021-12-31')

stocks = [data_nvda, data_gme, data_dji, data_ma]
prepped_stocks = []

for stock in stocks:
    prepped_stocks.append(preprocess(stock))

# nvda_ten = torch.tensor(data_nvda, dtype=np.float64);
# gme_ten = torch.tensor(data_gme, dtype=np.float64);
# dji_ten = torch.tensor(data_dji, dtype=np.float64);
# ma_ten = torch.tensor(data_ma, dtype=np.float64);


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

data_close size:  (251, 1)


TypeError: DataLoader.__init__() got multiple values for argument 'batch_size'

2. Defining an RNN. Unlike the previous FNN models, we want to predict the prices in the future
N days based on the prices in the past M days (including today). Usually, M is much larger than N. Here,
you may simply set N to be 1 and choose an integer that is at least 50 for M. You are required to define a
neural network which has at least one recurrent layer for this purpose.

In [ ]:
num_prev_days = 75;
num_future_days = 1;

